# 03 · Validation Engine & Duplicate Detection

Two things happen here:

1. A small **rule-based validation engine** applies the rule catalog from Notebook 01 to a
   simulated raw student-level extract (range, domain, and cross-field consistency checks).
2. A **probabilistic record-linkage** pipeline finds near-duplicate student records in the
   synthetic roster using string-similarity blocking (Jaro-Winkler), scored in the spirit of the
   Fellegi–Sunter framework, and evaluated against the known ground truth from Notebook 01.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import jellyfish
from pathlib import Path
from itertools import combinations
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

DATA_DIR = Path("../data")
DOCS_DIR = Path("../docs")
PLOT_TEMPLATE = "plotly_white"

roster = pd.read_csv(DATA_DIR / "student_roster.csv")
rules = pd.read_csv(DATA_DIR / "validation_rules.csv")
print(f"Roster: {len(roster):,} records ({roster.is_injected_duplicate.sum()} known injected duplicates)")
rules

Roster: 3,150 records (150 known injected duplicates)


,rule_id,category,severity,description
0,R001,Completeness,Error,Required field missing
1,R002,Range,Error,Grade level outside valid range (PK-12)
2,R003,Domain,Error,Invalid disability classification code
3,R004,Consistency,Error,Exit date precedes entry date
4,R005,Consistency,Warning,SWD flag set but no IEP date on file
5,R006,Referential,Error,District ID not found in master file
6,R007,Referential,Warning,Student ID not found in enrollment extract
7,R008,Uniqueness,Warning,Duplicate student record within district
8,R009,Range,Error,Date of birth implies age outside 3-21
9,R010,Format,Error,Invalid BEDS code format


## 1. Rule-based validation engine

We simulate a raw extract with a handful of intentionally corrupted fields, then apply vectorized rule checks — the same pattern a production validation engine uses, just at small scale.

In [2]:
rng = np.random.default_rng(42)
N = 2000
extract = pd.DataFrame({
    "student_id": rng.integers(100000000, 999999999, size=N).astype(str),
    "grade_level": rng.integers(-1, 14, N),                          # -1 and 13 are invalid (valid: 0-12, PK=-1 code excluded intentionally to trigger R002)
    "disability_code": rng.choice(["LD","OHI","ASD","SLD","EMN","ZZZ","",None], N,
                                   p=[0.28,0.18,0.12,0.15,0.10,0.05,0.07,0.05]),
    "entry_date": pd.to_datetime("2024-09-01") + pd.to_timedelta(rng.integers(0, 600, N), unit="D"),
    "exit_date": pd.NaT,
    "birth_year": 2026 - rng.integers(2, 24, N),                     # some imply age >21 or <3
    "district_id_ref": rng.choice(pd.read_csv(DATA_DIR / "districts.csv").district_id.tolist() + ["D999999"], N,
                                   p=[0.99/60]*60 + [0.01]),
})
# inject some exit dates before entry dates (R004)
bad_exit_idx = rng.choice(N, size=int(N*0.02), replace=False)
extract.loc[bad_exit_idx, "exit_date"] = extract.loc[bad_exit_idx, "entry_date"] - pd.to_timedelta(rng.integers(5, 90, len(bad_exit_idx)), unit="D")

VALID_DISABILITY_CODES = {"LD","OHI","ASD","SLD","EMN","AUT","SLI","OHI-ADHD"}
KNOWN_DISTRICTS = set(pd.read_csv(DATA_DIR / "districts.csv").district_id)

def check_r001(df): return df.disability_code.isna() | (df.disability_code == "")
def check_r002(df): return ~df.grade_level.between(0, 12)
def check_r003(df): return ~df.disability_code.isin(VALID_DISABILITY_CODES) & df.disability_code.notna() & (df.disability_code != "")
def check_r004(df): return df.exit_date.notna() & (df.exit_date < df.entry_date)
def check_r006(df): return ~df.district_id_ref.isin(KNOWN_DISTRICTS)
def check_r009(df): return ~(2026 - df.birth_year).between(3, 21)

engine = {
    "R001": check_r001, "R002": check_r002, "R003": check_r003,
    "R004": check_r004, "R006": check_r006, "R009": check_r009,
}

results = {}
for rule_id, fn in engine.items():
    flags = fn(extract)
    results[rule_id] = dict(
        rule=rules.set_index("rule_id").loc[rule_id, "description"],
        severity=rules.set_index("rule_id").loc[rule_id, "severity"],
        violations=int(flags.sum()),
        rate=float(flags.mean()),
    )

engine_results = pd.DataFrame(results).T
engine_results.index.name = "rule_id"
engine_results = engine_results.reset_index()
engine_results["violations"] = engine_results.violations.astype(int)
engine_results["rate"] = engine_results.rate.astype(float)
engine_results.sort_values("violations", ascending=False)

,rule_id,rule,severity,violations,rate
1,R002,Grade level outside valid range (PK-12),Error,270,0.1350
5,R009,Date of birth implies age outside 3-21,Error,263,0.1315
0,R001,Required field missing,Error,235,0.1175
2,R003,Invalid disability classification code,Error,91,0.0455
3,R004,Exit date precedes entry date,Error,40,0.0200
4,R006,District ID not found in master file,Error,20,0.0100


In [3]:
fig_engine = px.bar(engine_results.sort_values("violations"), x="violations", y="rule_id",
                     orientation="h", color="severity", text="violations",
                     color_discrete_map={"Error": "#D64550", "Warning": "#E8A33D"},
                     template=PLOT_TEMPLATE, title=f"Validation Engine Results on {N:,}-Row Simulated Extract")
fig_engine.update_layout(height=380, xaxis_title="Records flagged", yaxis_title=None)
fig_engine.show()

## 2. Probabilistic record linkage — duplicate student detection

**Blocking**: only compare pairs within the same district (comparing all pairs statewide would be O(n²) on ~3,000 records — blocking cuts that drastically).

**Scoring**: a Fellegi–Sunter-style weighted score combining Jaro-Winkler similarity on first/last name, exact-match on date of birth, and exact-match on district.

In [4]:
def candidate_pairs(df):
    pairs = []
    for _, grp in df.groupby("district_id"):
        idx = grp.index.tolist()
        if len(idx) > 400:   # cap block size for runtime; large blocks are rare here
            continue
        for i, j in combinations(idx, 2):
            pairs.append((i, j))
    return pairs

pairs = candidate_pairs(roster)
print(f"Candidate pairs after blocking by district: {len(pairs):,}")

def score_pair(a, b):
    fn_sim = jellyfish.jaro_winkler_similarity(str(a.first_name), str(b.first_name))
    ln_sim = jellyfish.jaro_winkler_similarity(str(a.last_name), str(b.last_name))
    dob_match = 1.0 if a.dob == b.dob else 0.0
    grade_match = 1.0 if a.grade == b.grade else 0.0
    # Fellegi-Sunter-style additive weights (agreement weight > disagreement weight per field)
    score = 3.0*fn_sim + 3.0*ln_sim + 2.5*dob_match + 0.5*grade_match
    return score, fn_sim, ln_sim, dob_match

rows = []
for i, j in pairs:
    a, b = roster.loc[i], roster.loc[j]
    score, fn_sim, ln_sim, dob_match = score_pair(a, b)
    rows.append(dict(idx_a=i, idx_b=j, student_a=a.student_id, student_b=b.student_id,
                      name_a=f"{a.first_name} {a.last_name}", name_b=f"{b.first_name} {b.last_name}",
                      fn_sim=fn_sim, ln_sim=ln_sim, dob_match=dob_match, score=score,
                      true_dupe=(a.duplicate_of == b.student_id) or (b.duplicate_of == a.student_id)))

scored = pd.DataFrame(rows)
print(f"Scored {len(scored):,} candidate pairs. Known true duplicate pairs among them: {scored.true_dupe.sum()}")
scored.sort_values("score", ascending=False).head(8)

Candidate pairs after blocking by district: 150,059


Scored 150,059 candidate pairs. Known true duplicate pairs among them: 127


,idx_a,idx_b,student_a,student_b,name_a,name_b,fn_sim,ln_sim,dob_match,score,true_dupe
150032,2951,2969,977064153,791560188,Emma Lewis,Emma Lewis,1.0,1.0,1.0,9.0,True
89482,1118,1657,998009098,707475242,Matthew Ramirez,Matthew Ramirez,1.0,1.0,1.0,9.0,True
89349,934,1725,932099374,769463280,Chloe Jones,Chloe Jones,1.0,1.0,1.0,9.0,True
48044,1901,3147,747494666,947966126,Scarlett Jones,Scarlett Jones,1.0,1.0,1.0,9.0,True
89209,882,1933,970349104,747288534,Scarlett Clark,Scarlett Clark,1.0,1.0,1.0,9.0,True
114476,1013,2717,937411091,760931218,Michael Moore,Michael Moore,1.0,1.0,1.0,9.0,True
63940,1140,2487,904337135,712285828,Lucas Thompson,Lucas Thompson,1.0,1.0,1.0,9.0,True
81282,1105,3137,721332448,992775293,Henry Lopez,Henry Lopez,1.0,1.0,1.0,9.0,True


In [5]:
fig_hist = px.histogram(scored, x="score", color="true_dupe", nbins=40, barmode="overlay",
                         template=PLOT_TEMPLATE, opacity=0.75,
                         color_discrete_map={True: "#D64550", False: "#9AA6C3"},
                         labels={"true_dupe": "Known duplicate"},
                         title="Match-Score Distribution: Known Duplicates vs. Non-Duplicates")
fig_hist.add_vline(x=7.0, line_dash="dot", line_color="#16213E", annotation_text="threshold = 7.0")
fig_hist.update_layout(height=420)
fig_hist.show()

In [6]:
THRESHOLD = 7.0
scored["predicted_dupe"] = scored.score >= THRESHOLD

precision = precision_score(scored.true_dupe, scored.predicted_dupe)
recall = recall_score(scored.true_dupe, scored.predicted_dupe)
f1 = f1_score(scored.true_dupe, scored.predicted_dupe)
cm = confusion_matrix(scored.true_dupe, scored.predicted_dupe)

print(f"Threshold = {THRESHOLD}")
print(f"Precision: {precision:.3f}   Recall: {recall:.3f}   F1: {f1:.3f}")
print()
print("Confusion matrix [rows=actual, cols=predicted], labels=[Not dupe, Dupe]:")
print(cm)

Threshold = 7.0


Precision: 0.982   Recall: 0.882   F1: 0.929

Confusion matrix [rows=actual, cols=predicted], labels=[Not dupe, Dupe]:
[[149930      2]
 [    15    112]]


### Threshold sensitivity

Sweeping the match threshold shows the classic precision/recall trade-off — useful for choosing an operating point (e.g., auto-merge above a high threshold, flag-for-review in a middle band).

In [7]:
thresholds = np.arange(4, 9, 0.25)
sweep = []
for t in thresholds:
    pred = scored.score >= t
    p = precision_score(scored.true_dupe, pred, zero_division=0)
    r = recall_score(scored.true_dupe, pred, zero_division=0)
    f = f1_score(scored.true_dupe, pred, zero_division=0)
    sweep.append((t, p, r, f))
sweep_df = pd.DataFrame(sweep, columns=["threshold", "precision", "recall", "f1"])

fig_sweep = go.Figure()
for col, color in zip(["precision", "recall", "f1"], ["#12A594", "#D64550", "#16213E"]):
    fig_sweep.add_trace(go.Scatter(x=sweep_df.threshold, y=sweep_df[col], mode="lines+markers",
                                    name=col.capitalize(), line=dict(color=color, width=2)))
fig_sweep.update_layout(template=PLOT_TEMPLATE, height=380, title="Precision / Recall / F1 vs. Match Threshold",
                         xaxis_title="Score threshold", yaxis_title="Metric value", yaxis_tickformat=".0%")
fig_sweep.show()

best = sweep_df.loc[sweep_df.f1.idxmax()]
print(f"Best F1 = {best.f1:.3f} at threshold = {best.threshold}")

Best F1 = 0.958 at threshold = 6.5


## 3. Export static dashboard page

In [8]:
import json
NAV_PAGES = json.load(open("../config/nav_pages.json"))

def render_nav(active):
    links = "\n".join(
        f'<a href="{href}" class="{"active" if key == active else ""}">{label}</a>'
        for key, label, href in NAV_PAGES
    )
    return f'''<nav class="site-nav"><div class="nav-inner">
      <div class="brand"><span class="dot"></span> NYSED Data Integrity Platform</div>
      <div class="links">{links}</div>
    </div></nav>'''

def page_shell(title, eyebrow, description, body_html, active, filename):
    html = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{title} · NYSED Data Integrity Platform</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Lora:wght@500;600;700&family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;600;700&display=swap" rel="stylesheet">
<link rel="stylesheet" href="../assets/style.css">
</head><body>
{render_nav(active)}
<header class="dash-header"><div class="dash-header-inner">
  <div class="eyebrow">{eyebrow}</div>
  <h1>{title}</h1>
  <p>{description}</p>
</div></header>
<main class="dash-body">
{body_html}
</main>
<footer class="site-footer">
  Synthetic demonstration data · NYSED Data Integrity &amp; Risk Monitoring Platform ·
  <a href="https://github.com/zia207/NYSED-Data-Integrity" target="_blank">Source code</a>
</footer>
</body></html>'''
    out = DOCS_DIR / "dashboard" / filename
    out.write_text(html)
    print("wrote", out)

def kpi_card(label, value, delta=None, tone="good", warn=False):
    cls = "kpi-card" + (" warn" if warn else "")
    delta_html = f'<div class="kpi-delta {tone}">{delta}</div>' if delta else ""
    return f'<div class="{cls}"><div class="kpi-label">{label}</div><div class="kpi-value">{value}</div>{delta_html}</div>'

def fig_div(f, div_id, include_js=False):
    return pio.to_html(f, include_plotlyjs="inline" if include_js else False, full_html=False, div_id=div_id)

In [9]:
kpi_html = f'''<div class="kpi-row">
  {kpi_card("Records Validated", f"{N:,}")}
  {kpi_card("Rule Violations Found", f"{int(engine_results.violations.sum()):,}", warn=True, tone="bad")}
  {kpi_card("Candidate Pairs Scored", f"{len(scored):,}")}
  {kpi_card("Duplicate Detection F1", f"{f1:.2f}", f"P={precision:.2f} · R={recall:.2f}")}
</div>'''

body = kpi_html + f'''
<div class="chart-panel">
  <h3>Rule engine results</h3>
  <div class="chart-note">Simulated {N:,}-row raw extract validated against the rule catalog from Notebook 01.</div>
  {fig_div(fig_engine, "fig-engine", include_js=True)}
</div>
<div class="chart-row">
  <div class="chart-panel">
    <h3>Match-score distribution</h3>
    <div class="chart-note">Fellegi–Sunter-style weighted score; red = known injected duplicates.</div>
    {fig_div(fig_hist, "fig-hist")}
  </div>
  <div class="chart-panel">
    <h3>Threshold sensitivity</h3>
    <div class="chart-note">Best F1 = {best.f1:.2f} at threshold {best.threshold}.</div>
    {fig_div(fig_sweep, "fig-sweep")}
  </div>
</div>
<div class="chart-panel">
  <h3>Validation rule catalog</h3>
  <table class="kpi-table"><thead><tr><th>Rule</th><th>Category</th><th>Severity</th><th>Description</th></tr></thead><tbody>
  {"".join(f"<tr><td class='mono'>{r.rule_id}</td><td>{r.category}</td><td><span class='pill {'pill-risk' if r.severity=='Error' else 'pill-ops'}'>{r.severity}</span></td><td>{r.description}</td></tr>" for r in rules.itertuples())}
  </tbody></table>
</div>
'''

page_shell(
    title="Validation &amp; Duplicate Detection",
    eyebrow="Notebook 03",
    description="A rule-based validation engine plus probabilistic record linkage (Jaro-Winkler + Fellegi-Sunter-style scoring) for near-duplicate student records.",
    body_html=body, active="validation.html", filename="validation.html",
)

wrote ../docs/dashboard/validation.html
